# 02 — Baseline SD1.5 Inpainting (Config A)

Notebook chạy baseline **thuần** (không ControlNet, không IP-Adapter, không LoRA):
- **Model:** `runwayml/stable-diffusion-inpainting` (SD1.5)
- **Dataset:** cùng `synthetic_occ` như `full-eval.ipynb`
- **Test set:** 666 ảnh/bin × 3 bins (20–40%, 40–60%, 60–80%), `random_state=42` → **1.998 ảnh**
- **Metrics:** L1, L2, ICP, SS, PSNR, SSIM, LPIPS, FID

**Kaggle:** chỉnh `DATASET_ROOT` trong cell config nếu tên dataset khác.


In [ ]:
import sys
!{sys.executable} -m pip install -q \
    "diffusers>=0.25.0" transformers accelerate xformers \
    scikit-image lpips clean-fid opencv-python-headless \
    huggingface_hub tqdm
print("Dependencies ready")


In [ ]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Dataset (Kaggle) — giống full-eval.ipynb ─────────────────────────────────
DATASET_ROOT = Path("/kaggle/input/datasets/dangvy1507/vehicle")
SYNTH_DIR    = DATASET_ROOT / "synthetic_occ"
GT_DIR       = SYNTH_DIR / "x_gt"
OCC_DIR      = SYNTH_DIR / "x_occ"
MASK_DIR     = SYNTH_DIR / "masks"
META_CSV     = SYNTH_DIR / "metadata_synthetic_occ.csv"

BASELINE_DIR  = Path("/kaggle/working/baseline_sd15_inpaint")
PRED_DIR      = BASELINE_DIR / "x_hat"
EVAL_GT_DIR   = BASELINE_DIR / "fid_gt"
EVAL_PRED_DIR = BASELINE_DIR / "fid_pred"
REPORT_DIR    = BASELINE_DIR / "reports"

for d in [BASELINE_DIR, PRED_DIR, EVAL_GT_DIR, EVAL_PRED_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32
MODEL_ID = "runwayml/stable-diffusion-inpainting"

PROMPT = "a car, realistic, high quality, detailed, complete, no occlusion"
NEG_PROMPT = "blurry, distorted, artifacts, extra car, duplicate, oversaturated"
NUM_STEPS = 20
GUIDANCE = 7.5
BATCH_SIZE = 4 if DEVICE == "cuda" else 1

# Stratified như full-eval: 666 × 3 = 1998
TEST_SIZE = 2000
BIN_EDGES  = [(0.20, 0.40), (0.40, 0.60), (0.60, 0.80)]
BIN_LABELS = ["20-40%", "40-60%", "60-80%"]

assert SYNTH_DIR.exists(), f"Không tìm thấy dataset: {SYNTH_DIR}"
assert META_CSV.exists(),  f"Không tìm thấy metadata: {META_CSV}"
assert GT_DIR.exists(),    f"Không tìm thấy x_gt: {GT_DIR}"

meta = pd.read_csv(META_CSV)
meta.columns = meta.columns.str.strip().str.lower()

print(f"Meta rows: {len(meta):,}")
print(f"Device: {DEVICE}")
print(f"SYNTH_DIR: {SYNTH_DIR}")
print(f"BASELINE_DIR: {BASELINE_DIR}")
print(f"Model: {MODEL_ID}")
print(f"TEST_SIZE target: {TEST_SIZE}")
print(f"Dataset OK — {len(list(GT_DIR.iterdir())):,} files in x_gt")


In [ ]:
from diffusers import StableDiffusionInpaintPipeline, PNDMScheduler, DPMSolverMultistepScheduler

SCHEDULER_NAME = "dpm"

pipe = StableDiffusionInpaintPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    safety_checker=None,
    requires_safety_checker=False,
)

if SCHEDULER_NAME.lower() == "pndm":
    pipe.scheduler = PNDMScheduler.from_config(pipe.scheduler.config)
else:
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

pipe = pipe.to(DEVICE)

xformers_ok = False
try:
    pipe.enable_xformers_memory_efficient_attention()
    xformers_ok = True
except Exception:
    pipe.enable_attention_slicing("auto")

pipe.vae.enable_slicing()

print(f"Loaded: {MODEL_ID}")
print(f"Scheduler: {pipe.scheduler.__class__.__name__}")
print(f"xFormers: {xformers_ok}")


In [ ]:
from PIL import Image


def inpaint(image: Image.Image, mask: Image.Image, prompt: str, steps: int = 20,
            negative_prompt: str = "", guidance_scale: float = 7.5, seed: int = 42) -> Image.Image:
    generator = torch.Generator(device=DEVICE).manual_seed(seed)
    with torch.no_grad():
        if DEVICE == "cuda":
            with torch.autocast("cuda"):
                out = pipe(
                    prompt=prompt,
                    negative_prompt=negative_prompt,
                    image=image,
                    mask_image=mask,
                    num_inference_steps=steps,
                    guidance_scale=guidance_scale,
                    generator=generator,
                ).images[0]
        else:
            out = pipe(
                prompt=prompt,
                negative_prompt=negative_prompt,
                image=image,
                mask_image=mask,
                num_inference_steps=steps,
                guidance_scale=guidance_scale,
                generator=generator,
            ).images[0]
    return out


def make_test_split_stratified(meta_df: pd.DataFrame, test_size: int, bin_edges, seed: int = 42):
    per_bin = test_size // len(bin_edges)
    parts = []
    for lo, hi in bin_edges:
        sub = meta_df[(meta_df["occlusion_ratio"] >= lo) & (meta_df["occlusion_ratio"] < hi)]
        parts.append(sub.sample(min(per_bin, len(sub)), random_state=seed))
    test_df = pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)

    def assign_bin(r):
        for (lo, hi), lbl in zip(bin_edges, BIN_LABELS):
            if lo <= r < hi:
                return lbl
        return "?"

    test_df["bin_label"] = test_df["occlusion_ratio"].apply(assign_bin)
    return test_df


test_df = make_test_split_stratified(meta, TEST_SIZE, BIN_EDGES, SEED)
print(f"Test set: {len(test_df):,} images (≈{TEST_SIZE // len(BIN_EDGES)}/bin × {len(BIN_EDGES)} bins)")
for (lo, hi), lbl in zip(BIN_EDGES, BIN_LABELS):
    n = len(test_df[(test_df["occlusion_ratio"] >= lo) & (test_df["occlusion_ratio"] < hi)])
    print(f"  {lbl}: {n}")


In [ ]:
import matplotlib.pyplot as plt

PROMPT_TRIALS = [
    {"name": "full_eval_style", "prompt": PROMPT, "negative": NEG_PROMPT},
    {"name": "texture_plus", "prompt": "a realistic car, high quality, detailed paint texture, natural lighting",
     "negative": "blurry, plastic texture, oversaturated, artifacts"},
]

sample_row = test_df.sample(1, random_state=SEED).iloc[0]
img_occ  = Image.open(OCC_DIR  / sample_row["x_occ"]).convert("RGB").resize((512, 512))
img_mask = Image.open(MASK_DIR / sample_row["mask"]).convert("L").resize((512, 512))
img_gt   = Image.open(GT_DIR   / sample_row["x_gt"]).convert("RGB").resize((512, 512))

fig, axes = plt.subplots(1, len(PROMPT_TRIALS) + 2, figsize=(4 * (len(PROMPT_TRIALS) + 2), 4))
axes[0].imshow(img_occ); axes[0].set_title("x_occ")
axes[1].imshow(img_mask, cmap="gray"); axes[1].set_title("mask")
for i, cfg in enumerate(PROMPT_TRIALS, start=2):
    pred = inpaint(img_occ, img_mask, cfg["prompt"], steps=NUM_STEPS,
                   negative_prompt=cfg["negative"], guidance_scale=GUIDANCE, seed=SEED)
    axes[i].imshow(pred); axes[i].set_title(cfg["name"])
for ax in axes:
    ax.axis("off")
plt.tight_layout(); plt.show()

PROMPT = PROMPT_TRIALS[0]["prompt"]
NEG_PROMPT = PROMPT_TRIALS[0]["negative"]
print("Using prompt:", PROMPT)
print("Using negative:", NEG_PROMPT)


In [ ]:
from tqdm.auto import tqdm
import time


def run_baseline_inference(df: pd.DataFrame, batch_size: int = 4):
    rows = []
    t0 = time.perf_counter()

    for p in EVAL_GT_DIR.glob("*.png"):
        p.unlink()
    for p in EVAL_PRED_DIR.glob("*.png"):
        p.unlink()

    for i in tqdm(range(0, len(df), batch_size), desc="Baseline inference"):
        chunk = df.iloc[i:i + batch_size]
        images, masks, stems = [], [], []
        for _, r in chunk.iterrows():
            images.append(Image.open(OCC_DIR / r["x_occ"]).convert("RGB").resize((512, 512)))
            masks.append(Image.open(MASK_DIR / r["mask"]).convert("L").resize((512, 512)))
            stems.append(r["stem"])

        prompts = [PROMPT] * len(images)
        negs = [NEG_PROMPT] * len(images)
        generators = [torch.Generator(device=DEVICE).manual_seed(SEED + i + j) for j in range(len(images))]

        with torch.no_grad():
            if DEVICE == "cuda":
                with torch.autocast("cuda"):
                    outs = pipe(
                        prompt=prompts, negative_prompt=negs, image=images, mask_image=masks,
                        num_inference_steps=NUM_STEPS, guidance_scale=GUIDANCE, generator=generators,
                    ).images
            else:
                outs = pipe(
                    prompt=prompts, negative_prompt=negs, image=images, mask_image=masks,
                    num_inference_steps=NUM_STEPS, guidance_scale=GUIDANCE, generator=generators,
                ).images

        for (_, r), pred, stem in zip(chunk.iterrows(), outs, stems):
            gt = Image.open(GT_DIR / r["x_gt"]).convert("RGB").resize((512, 512))
            pred_path = PRED_DIR / f"{stem}.png"
            pred.save(pred_path)
            gt.save(EVAL_GT_DIR / f"{stem}.png")
            pred.save(EVAL_PRED_DIR / f"{stem}.png")

            rows.append({
                "stem": stem,
                "x_gt": r["x_gt"],
                "x_occ": r["x_occ"],
                "mask": r["mask"],
                "pred": pred_path.name,
                "occlusion_ratio": float(r["occlusion_ratio"]),
                "bin_label": r.get("bin_label", "?"),
            })

    elapsed = time.perf_counter() - t0
    return pd.DataFrame(rows), elapsed


pred_df, infer_elapsed = run_baseline_inference(test_df, batch_size=BATCH_SIZE)
pred_df.to_csv(REPORT_DIR / "baseline_predictions.csv", index=False)
print(f"Done: {len(pred_df):,} images | {infer_elapsed/60:.2f} min")


In [ ]:
import cv2
from PIL import Image
import lpips
from skimage.metrics import structural_similarity as ssim, peak_signal_noise_ratio as psnr_fn
from cleanfid import fid as cleanfid
from torchvision import transforms
from torchvision.models import inception_v3
from torchvision.models.segmentation import deeplabv3_resnet101

lpips_model = lpips.LPIPS(net="alex").to(DEVICE)
lpips_model.eval()


def pixel_l1(gt, pred, mask01):
    gt_f = gt.astype(np.float64) / 255.0
    pr_f = pred.astype(np.float64) / 255.0
    m = mask01.astype(bool)
    return float(np.mean(np.abs(gt_f[m] - pr_f[m]))) if m.sum() > 0 else float(np.mean(np.abs(gt_f - pr_f)))


def pixel_l2(gt, pred, mask01):
    gt_f = gt.astype(np.float64) / 255.0
    pr_f = pred.astype(np.float64) / 255.0
    m = mask01.astype(bool)
    return float(np.mean((gt_f[m] - pr_f[m]) ** 2)) if m.sum() > 0 else float(np.mean((gt_f - pr_f) ** 2))


def masked_psnr(gt_rgb, pr_rgb, mask01):
    m = mask01.astype(bool)
    gt_m = gt_rgb[m].astype(np.float64)
    pr_m = pr_rgb[m].astype(np.float64)
    mse = np.mean((gt_m - pr_m) ** 2) if m.sum() > 0 else np.mean(
        (gt_rgb.astype(np.float64) - pr_rgb.astype(np.float64)) ** 2
    )
    return float(10 * np.log10(255.0 ** 2 / mse)) if mse > 0 else 100.0


def masked_ssim(gt_rgb, pred_rgb, mask01):
    gt_f = gt_rgb.astype(np.float32) / 255.0
    pr_f = pred_rgb.astype(np.float32) / 255.0
    _, ssim_map = ssim(gt_f, pr_f, channel_axis=2, data_range=1.0, full=True)
    m = mask01.astype(bool)
    return float(ssim_map[m].mean()) if m.sum() > 0 else float(ssim_map.mean())


def masked_lpips(gt_rgb, pred_rgb, mask01):
    m = mask01.astype(np.float32)[..., None]
    gt_m = (gt_rgb.astype(np.float32) * m).astype(np.uint8)
    pr_m = (pred_rgb.astype(np.float32) * m).astype(np.uint8)
    gt_t = torch.from_numpy(gt_m).permute(2, 0, 1).unsqueeze(0).float() / 127.5 - 1.0
    pr_t = torch.from_numpy(pr_m).permute(2, 0, 1).unsqueeze(0).float() / 127.5 - 1.0
    with torch.no_grad():
        return float(lpips_model(gt_t.to(DEVICE), pr_t.to(DEVICE)).item())


print("Loading Inception V3 (ICP) ...")
try:
    from torchvision.models import Inception_V3_Weights
    inception_net = inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1)
except (ImportError, AttributeError):
    inception_net = inception_v3(pretrained=True)
inception_net.eval().to(DEVICE)

IMAGENET_CAR_CLASSES = [407, 436, 511, 627, 656, 705, 717, 734, 751, 779, 817, 820, 868]
inception_tf = transforms.Compose([
    transforms.Resize(299), transforms.CenterCrop(299),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


def compute_icp(pred_pil, mask_np_gray):
    ys, xs = np.where(mask_np_gray > 127)
    if len(ys) == 0:
        return 0.0
    pad = 16
    h, w = mask_np_gray.shape
    y0 = max(0, int(ys.min()) - pad)
    y1 = min(h, int(ys.max()) + pad)
    x0 = max(0, int(xs.min()) - pad)
    x1 = min(w, int(xs.max()) + pad)
    crop = pred_pil.crop((x0, y0, x1, y1))
    if crop.width < 10 or crop.height < 10:
        return 0.0
    inp = inception_tf(crop.convert("RGB")).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        out = inception_net(inp)
        logits = out.logits if hasattr(out, "logits") else out
        probs = torch.softmax(logits, dim=1)[0].cpu()
    return float(probs[IMAGENET_CAR_CLASSES].sum())


print("Loading DeepLab V3 (SS) ...")
try:
    from torchvision.models.segmentation import DeepLabV3_ResNet101_Weights
    deeplab_net = deeplabv3_resnet101(weights=DeepLabV3_ResNet101_Weights.COCO_WITH_VOC_LABELS_V1)
except (ImportError, AttributeError):
    deeplab_net = deeplabv3_resnet101(pretrained=True)
deeplab_net.eval().to(DEVICE)

CAR_CLASS_IDX = 7
seg_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


def compute_ss(pred_pil, mask01):
    inp = seg_tf(pred_pil.convert("RGB")).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        seg_out = deeplab_net(inp)["out"][0]
    pred_seg = seg_out.argmax(0).cpu().numpy().astype(np.uint8)
    pred_seg = cv2.resize(pred_seg, (512, 512), interpolation=cv2.INTER_NEAREST)
    m = mask01.astype(bool)
    return float((pred_seg[m] == CAR_CLASS_IDX).mean()) if m.sum() > 0 else 0.0


ALL_METRICS = ["l1", "l2", "icp", "ss", "psnr", "ssim", "lpips"]


def bin_summary(df, lo, hi, label):
    sub = df[(df["occlusion_ratio"] >= lo) & (df["occlusion_ratio"] < hi)]
    if len(sub) == 0:
        return {"bin": label, "n": 0, **{m: None for m in ALL_METRICS}}
    row = {"bin": label, "n": len(sub)}
    for m in ALL_METRICS:
        row[m] = round(float(sub[m].mean()), 4)
    return row


metric_rows = []
for _, r in tqdm(pred_df.iterrows(), total=len(pred_df), desc="Metrics"):
    gt_bgr = cv2.imread(str(GT_DIR / r["x_gt"]))
    pr_bgr = cv2.imread(str(PRED_DIR / r["pred"]))
    if gt_bgr is None or pr_bgr is None:
        continue
    gt = cv2.resize(cv2.cvtColor(gt_bgr, cv2.COLOR_BGR2RGB), (512, 512))
    pr = cv2.resize(cv2.cvtColor(pr_bgr, cv2.COLOR_BGR2RGB), (512, 512))
    mk_raw = cv2.imread(str(MASK_DIR / r["mask"]), cv2.IMREAD_GRAYSCALE)
    mk = cv2.resize(mk_raw, (512, 512))
    mk01 = (mk > 127).astype(np.uint8)
    pr_pil = Image.fromarray(pr)

    metric_rows.append({
        "stem": r["stem"],
        "occlusion_ratio": r["occlusion_ratio"],
        "bin_label": r.get("bin_label", "?"),
        "l1": pixel_l1(gt, pr, mk01),
        "l2": pixel_l2(gt, pr, mk01),
        "icp": compute_icp(pr_pil, mk),
        "ss": compute_ss(pr_pil, mk01),
        "psnr": masked_psnr(gt, pr, mk01),
        "ssim": masked_ssim(gt, pr, mk01),
        "lpips": masked_lpips(gt, pr, mk01),
    })

metric_df = pd.DataFrame(metric_rows)
fid_value = float(cleanfid.compute_fid(str(EVAL_GT_DIR), str(EVAL_PRED_DIR), mode="clean"))
print(f"FID = {fid_value:.2f}")

overall = {"bin": "overall", "n": len(metric_df)}
for m in ALL_METRICS:
    overall[m] = round(float(metric_df[m].mean()), 4)

summary_rows = [overall]
for (lo, hi), lbl in zip(BIN_EDGES, BIN_LABELS):
    summary_rows.append(bin_summary(metric_df, lo, hi, lbl))
summary_df = pd.DataFrame(summary_rows)
summary_df.insert(summary_df.columns.get_loc("psnr") + 3, "fid", [round(fid_value, 2)] + [None] * len(BIN_EDGES))

print("=== Summary (SD1.5 baseline, masked region) ===")
print(summary_df.to_string(index=False))

summary = {
    "config": "A_SD15_inpaint_only",
    "model": MODEL_ID,
    "scheduler": pipe.scheduler.__class__.__name__,
    "num_test_images": int(len(metric_df)),
    "prompt": PROMPT,
    "negative_prompt": NEG_PROMPT,
    "steps": int(NUM_STEPS),
    "guidance": float(GUIDANCE),
    "fid": fid_value,
    "inference_minutes": float(infer_elapsed / 60.0),
}
for m in ALL_METRICS:
    summary[f"{m}_mean"] = float(metric_df[m].mean())

metric_df.to_csv(REPORT_DIR / "baseline_metrics_per_image.csv", index=False)
summary_df.to_csv(REPORT_DIR / "baseline_summary_table.csv", index=False)
pd.DataFrame([summary]).to_csv(REPORT_DIR / "baseline_summary.csv", index=False)

print("\nSaved:", REPORT_DIR / "baseline_summary_table.csv")


In [ ]:
import matplotlib.pyplot as plt

analysis_df = metric_df.copy()
analysis_df["score"] = analysis_df["ssim"] - analysis_df["lpips"]
best20  = analysis_df.sort_values("score", ascending=False).head(20)
worst20 = analysis_df.sort_values("score", ascending=True).head(20)
best20.to_csv(REPORT_DIR  / "best20_cases.csv",  index=False)
worst20.to_csv(REPORT_DIR / "worst20_cases.csv", index=False)


def show_cases(case_df: pd.DataFrame, title: str, n_show: int = 10):
    n = min(n_show, len(case_df))
    fig, axes = plt.subplots(n, 4, figsize=(14, 3 * n))
    if n == 1:
        axes = np.expand_dims(axes, axis=0)

    for i, (_, row) in enumerate(case_df.head(n).iterrows()):
        stem = row["stem"]
        gt   = Image.open(EVAL_GT_DIR   / f"{stem}.png").convert("RGB")
        pr   = Image.open(EVAL_PRED_DIR / f"{stem}.png").convert("RGB")
        rr   = pred_df[pred_df["stem"] == stem].iloc[0]
        occ  = Image.open(OCC_DIR  / rr["x_occ"]).convert("RGB")
        mask = Image.open(MASK_DIR / rr["mask"]).convert("L")

        axes[i, 0].imshow(gt)
        axes[i, 0].set_title("x_gt")
        axes[i, 1].imshow(occ)
        axes[i, 1].set_title("x_occ")
        axes[i, 2].imshow(mask, cmap="gray")
        axes[i, 2].set_title("M")
        axes[i, 3].imshow(pr)
        axes[i, 3].set_title(
            f"x_hat\nSSIM={row['ssim']:.3f}, LPIPS={row['lpips']:.3f}, PS={row['psnr']:.1f}"
        )
        for j in range(4):
            axes[i, j].axis("off")

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


show_cases(best20,  "Best cases",  n_show=10)
show_cases(worst20, "Worst cases", n_show=10)


In [ ]:
ablation_row = {
    "config": "A_SD15_inpaint_only",
    "model": MODEL_ID,
    "l1_mean": summary["l1_mean"],
    "l2_mean": summary["l2_mean"],
    "icp_mean": summary["icp_mean"],
    "ss_mean": summary["ss_mean"],
    "psnr_mean": summary["psnr_mean"],
    "ssim_mean": summary["ssim_mean"],
    "lpips_mean": summary["lpips_mean"],
    "FID": summary["fid"],
    "num_test": summary["num_test_images"],
    "steps": summary["steps"],
    "guidance": summary["guidance"],
    "scheduler": summary["scheduler"],
}

pd.DataFrame([ablation_row]).to_csv(REPORT_DIR / "ablation_row_baseline.csv", index=False)
print("Saved:", REPORT_DIR / "ablation_row_baseline.csv")
pd.DataFrame([ablation_row])


In [ ]:
import subprocess

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    r = test_df.iloc[0]
    img_occ  = Image.open(OCC_DIR  / r["x_occ"]).convert("RGB").resize((512, 512))
    img_mask = Image.open(MASK_DIR / r["mask"]).convert("L").resize((512, 512))

    _ = inpaint(img_occ, img_mask, PROMPT, steps=10,
                negative_prompt=NEG_PROMPT, guidance_scale=GUIDANCE, seed=SEED)

    peak_mb = torch.cuda.max_memory_allocated() / 1024**2
    print(f"Peak VRAM (10 steps, 1 image): {peak_mb:.0f} MiB ({peak_mb/1024:.2f} GiB)")
    print(f"xFormers enabled: {xformers_ok}")

    try:
        smi = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,memory.total,memory.used,memory.free",
             "--format=csv,noheader,nounits"],
            stderr=subprocess.DEVNULL,
        ).decode().strip()
        print("nvidia-smi:\n" + smi)
    except Exception:
        print("Không đọc được nvidia-smi")
else:
    print("No CUDA GPU")
